# 2024 유로 스페인 빌드업 패턴: 구역 기반 전진 경로

2024 UEFA 유로에서 스페인이 치른 7경기 전체의 구역 기반 전진 경로를 그려, 경기별로 빌드업이 어떤 구역을 거쳐 전진했는지, 대회를 관통하는 스타일은 무엇이었는지 살펴봅니다.

- 피치를 mplsoccer의 `Pitch(positional=True)` 표준 Juego de Posición(포지션 플레이) 그리드(30구역, 가로 6단 x 세로 5채널)로 나눕니다.
- 구역 배경 음영으로 30구역별 패스 시작 위치 점유(방향 무관)를, 그 위 화살표로 구역 간 "전진" (더 앞선 가로단으로 넘어간) 패스 전환을 함께 표시합니다.
- 이 폴더(`zone_progression/`)는 "2024 유로 스페인의 빌드업 패턴" 주제의 구역 기반 전진 경로 방법론 전용 하위 폴더입니다. 분석 기획은 [`../PLAN.md`](../PLAN.md), 패스 네트워크 방법론은 [`../pass_network/`](../pass_network/), 백로그 항목은 `ideas/backlog.md`의 "2024 유로 스페인의 빌드업 패턴"을 참고하세요.

## 방법론: 구역 정의와 시각화

`plot_zone_progression()`(`src/visualizer.py`)의 계산 순서는 다음과 같습니다.

1. **구역 정의**: `pitch.dim.positional_x`/`positional_y`(StatsBomb 좌표계 기준 가로 `[0, 18, 39, 60, 81, 102, 120]`, 세로 `[0, 18, 30, 50, 62, 80]`)를 그대로 읽어와 30구역(6x5)으로 나눕니다. 가로단은 페널티박스 라인/하프라인에 맞춰 정의되고, 세로 채널은 하프스페이스를 포함합니다.
2. **점유(배경 음영)**: 성공한 패스의 시작 위치가 속한 구역별로 횟수를 세어, 30구역 전부(0회 포함)를 컬러맵으로 채웁니다. 방향과 무관하게 집계합니다.
3. **전진 전환(화살표)**: 도착 구역의 가로단 인덱스가 시작 구역보다 큰(더 앞선 가로단으로 넘어간) 성공 패스만 구역 쌍 단위로 집계하고, `min_transition_count`(기본 3) 이상 반복된 쌍만 화살표로 표시합니다. 화살표 두께는 반복 횟수에 비례합니다.

처음엔 직접 정한 3등분x5채널(15구역) 그리드와, 화살표 피치 + 별도 히트맵 패널로 나눈 2패널 구성으로 시작했습니다. 이후 두 가지 조정을 거쳤습니다.
- "포지션 플레이 전술을 반영할 수 없냐"는 요청으로, 직접 정한 격자 대신 mplsoccer의 `Pitch(positional=True)`가 그리는 표준 Juego de Posición 그리드(30구역)로 교체했습니다.
- "화살표(피치 좌표)와 히트맵(별도 격자)의 좌표계가 어긋나 보인다"는 지적으로, 두 레이어를 피치 하나 위에 통합했습니다(구역 배경 음영 + 화살표 오버레이).

**데이터 검토** (`scripts/review_zone_progression_data.py`, 7경기 전체 스페인 `Pass` 이벤트 기준): `location`/`pass_end_location` 결측 0건, `[x, y]` 형태가 아닌 이상값도 0건 — 전체 4334개 패스 모두 정상.

In [ ]:
import os
import sys

if sys.platform.startswith('win') and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

import matplotlib.pyplot as plt
from src.data_loader import get_competition_matches, get_match_events
from src.visualizer import plot_zone_progression

COMPETITION_ID = 55  # UEFA Euro
SEASON_ID = 282      # 2024
TEAM = "Spain"

output_dir = os.path.join(os.getcwd(), "processed", "spain_euro2024_zone_progression")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
matches = get_competition_matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
spain_matches = matches[(matches['home_team'] == TEAM) | (matches['away_team'] == TEAM)].copy()
spain_matches = spain_matches.sort_values('match_date')
spain_matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'competition_stage']]

In [ ]:
for _, match in spain_matches.iterrows():
    match_id = match['match_id']
    opponent = match['away_team'] if match['home_team'] == TEAM else match['home_team']
    stage = match['competition_stage']

    events = get_match_events(match_id=match_id)

    fig, ax = plot_zone_progression(
        events_df=events,
        team_name=TEAM,
        title=f"Spain Zone Progression - {stage} vs {opponent}",
    )

    filename = f"{stage.lower().replace(' ', '_')}_vs_{opponent.lower().replace(' ', '_')}.png"
    out_path = os.path.join(output_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
    plt.show()
    plt.close(fig)
    print(f"저장 완료: {out_path}")

## 관찰 기록

7경기 결과를 보며 경기별 차이와 대회 전체를 관통하는 스페인 빌드업 스타일을 정리한 결과는 `RESULTS.md`에 문서화할 예정입니다 (아직 미작성).